# Saved whole-image hazard evidence

CPU viewer for an existing `evaluation/hazard_report.json`. No evaluator, producers, models or downloads run here. Test is the default; development is separate. Synthetic fixtures are software checks, including scripted latency. Unscored phrases are not confirmed hallucinations. See `docs/hazard_visualization.md`.

In [ ]:
import json
import os
from pathlib import Path
import sys
REPO_ROOT_OVERRIDE = os.environ.get('TRAVERSABILITY_REPO_ROOT')
start = Path(REPO_ROOT_OVERRIDE or Path.cwd()).resolve()
candidates = [p for base in (start, *start.parents) for p in (base, base / 'VLM_evaluation')]
COMPONENT = next((p for p in candidates if (p / 'docs/hazard_interfaces.md').is_file()), None)
if COMPONENT is None:
    raise RuntimeError('Start within the checkout or set TRAVERSABILITY_REPO_ROOT.')
sys.path.insert(0, str(COMPONENT / 'src'))
from traversability_hazard_visualization import load_run, create_viewer, render_discovery, render_image, render_segmentation, render_deployment, export_figure
config = json.loads((COMPONENT / 'configs/hazards/visualization.json').read_text(encoding='utf-8'))
config['component_root'] = str(COMPONENT)
# Set explicit saved run_name/data_root/run_root here, or use TRAVERSABILITY_*_ROOT.
# Historical absolute report roots are ignored for asset loading.
SHOW_WIDGETS = True
EXPORT = False
FRAME_ID = None  # Exact opaque frame ID for the non-widget image render.
MODEL = None
CONDITION = None
PROFILE_IDS = []  # Explicit (component, condition_key, profile_id) tuples.


In [ ]:
saved = load_run(config)
print(json.dumps(saved.status(), indent=2))


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
figures = {'discovery': render_discovery(saved, split='test')}
if not SHOW_WIDGETS:
    figures.update(image=render_image(saved, FRAME_ID, model=MODEL, condition=CONDITION),
                   sam=render_segmentation(saved, split='test'),
                   deployment=render_deployment(saved, PROFILE_IDS))
    for figure in figures.values():
        display(figure)
        plt.close(figure)
else:
    plt.close(figures['discovery'])
    try:
        viewer = create_viewer(saved)
        display(viewer)
    except ImportError as error:
        print('Notebook/widget dependency unavailable:', error, '; use SHOW_WIDGETS=False.')
        display(figures['discovery'])


In [ ]:
if EXPORT:
    # Both switches are required; outputs stay in config['output_dir'].
    for name, figure in figures.items():
        print(export_figure(saved, figure, name + '.png', enabled=True))
else:
    print('Export disabled. Explicitly enable config["export_enabled"] and EXPORT to save PNG/SVG plus evidence sidecars.')
